# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

**Học viên:** Bùi Văn Quang (MSSV: 2A202602688)
**Chương trình:** VinUni AI Thực chiến - Track 4 (Computer Vision / Vision-Language Model)
**Nhiệm vụ:** Finetune Vision-Language Model `Qwen/Qwen3-VL-4B-Instruct` bằng QLoRA 4-bit NF4 trên Google Colab GPU Tesla T4 (16GB VRAM) để phát hiện và định vị các hư hại mặt đường (`pothole`, `crack`, `manhole`) với toạ độ grounding 0-1000.

Đọc kỹ `README.md`, `GUIDE.md` và `RUBRIC.md`. Notebook này được thiết kế hoàn chỉnh từ Phase 0 đến Phase 8.

## 0. Môi trường (Colab T4)

Kiểm tra GPU Tesla T4, cài đặt đúng phiên bản thư viện theo `requirements-colab.txt`, mount Google Drive và cấu hình Kaggle API qua Colab Secrets.

In [ ]:
# Runtime -> Change runtime type -> T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

import sys
import torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.2f} GB")
else:
    print("CẢNH BÁO: CUDA chưa được kích hoạt! Vui lòng vào Runtime -> Change runtime type -> T4 GPU.")


In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✓ Đã cấu hình Kaggle credentials từ Colab Secrets thành công.")
except Exception as e:
    print(f"Lưu ý: Không tìm thấy Colab Secrets ({e}).")
    print("Nếu tải dữ liệu bằng Kaggle API, vui lòng cấu hình KAGGLE_USERNAME và KAGGLE_KEY trong mục Secrets của Colab.")


In [ ]:
# Mount Google Drive và kiểm tra bộ test ban đầu
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/day10_checkpoints"
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    print(f"✓ Google Drive đã mount. Thư mục lưu checkpoint backup: {DRIVE_CHECKPOINT_DIR}")
except Exception as e:
    DRIVE_CHECKPOINT_DIR = "work/checkpoints"
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    print(f"Đang chạy cục bộ hoặc chưa mount Drive. Lưu checkpoint tại: {DRIVE_CHECKPOINT_DIR}")

# Chạy bộ test ban đầu để xác nhận công cụ chuẩn của đề bài hoạt động tốt
!python -m pytest -q


## 1. Dữ liệu: tải và chia split

Tải dataset Road Damage từ Kaggle (2.009 ảnh, 4.737 boxes) và chia split cố định theo phiên quay video (session-based) bằng `scripts/prepare_data.py` để chống rò rỉ dữ liệu giữa các frame liền kề.

In [ ]:
# Tải dữ liệu và chia split chuẩn hoá
!python scripts/prepare_data.py --kaggle


In [ ]:
# Kiểm tra thống kê split, kiểm tra session leakage và vẽ 6 ảnh ground truth mẫu
import json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from scripts.labcommon import read_jsonl, CLASSES
from src.grounding import box_to_pixel, draw_boxes_on_image

# 1. Đọc meta.json
meta_path = Path("data/splits/meta.json")
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print(f"Dataset Version: {meta.get('dataset_version')}")
    print(f"Seed: {meta.get('seed')} | Salt: {meta.get('salt')}")
    print("Phân bố ảnh và bounding boxes theo split:")
    for s, c in meta.get("counts", {}).items():
        print(f"  Split [{s:5s}]: {c['images']:5d} ảnh | Boxes: {c['boxes']}")

# 2. Kiểm tra Session Leakage giữa các split
session_splits = {}
for s in ["train", "val", "test"]:
    for r in read_jsonl(f"data/splits/{s}.jsonl"):
        session_splits.setdefault(r["session"], set()).add(s)

leakage = {sess: sp for sess, sp in session_splits.items() if len(sp) > 1}
assert len(leakage) == 0, f"Phát hiện rò rỉ phiên quay giữa các split: {leakage}"
print(f"✓ Session leakage check: ĐẠT! Tổng số {len(session_splits)} sessions phân chia độc lập hoàn toàn giữa train/val/test.")

# 3. Vẽ 6 ảnh ground truth mẫu từ tập train
train_rows = read_jsonl("data/splits/train.jsonl")
sample_indices = [0, 10, 25, 50, 100, 150]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=120)
axes = axes.flatten()
for i, idx in enumerate(sample_indices):
    entry = train_rows[idx % len(train_rows)]
    img_path = Path("data") / entry["image_path"]
    img = Image.open(img_path).convert("RGB")
    annotated = draw_boxes_on_image(img, entry["objects"], coord="norm1000", line_width=3)
    axes[i].imshow(annotated)
    axes[i].set_title(f"{entry['image_id']} ({len(entry['objects'])} boxes)", fontsize=9)
    axes[i].axis("off")
plt.tight_layout()
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000

Chuyển đổi nhãn từ YOLO `[xc, yc, w, h]` (chuẩn hoá [0, 1]) sang Grounding JSON `[x1, y1, x2, y2]` trên thang 0-1000, sắp xếp theo thứ tự `(y1, x1)` và kiểm tra tính đối xứng khi khôi phục pixel.

In [ ]:
# Kiểm tra các hàm chuyển đổi tọa độ và prompt grounding chuẩn
from src.grounding import yolo_to_box, objects_to_answer, box_to_pixel, pixel_to_box, DEFAULT_PROMPT

print("Grounding Prompt chuẩn:")
print(DEFAULT_PROMPT)

sample_objs = [
    {"label": "crack", "yolo": [0.5, 0.8, 0.2, 0.1]},
    {"label": "pothole", "yolo": [0.4, 0.3, 0.15, 0.15]},
]
ans_json = objects_to_answer(sample_objs)
print("\nGrounding JSON sinh ra (đã sắp xếp trên->dưới, trái->phải theo (y1, x1)):")
print(ans_json)

# Kiểm tra chuyển đổi pixel ngược
box = yolo_to_box("pothole", [0.5, 0.5, 0.2, 0.2])
px = box_to_pixel(box["bbox_2d"], width=640, height=360)
rec = pixel_to_box(px, width=640, height=360, label="pothole")
assert rec["bbox_2d"] == box["bbox_2d"], "Lệch tọa độ trong quá trình khôi phục pixel!"
print("✓ Kiểm tra chuyển đổi tọa độ YOLO <-> Grounding 0-1000: ĐẠT!")


## 3. E0: zero-shot

Nạp model gốc `Qwen/Qwen3-VL-4B-Instruct` ở chế độ 4-bit NF4 double quantization (`llm_int8_skip_modules=["visual", "lm_head"]`) và chạy suy luận zero-shot trên 311 ảnh test.

In [ ]:
import torch
from src.model_utils import load_model_and_processor, inspect_model_structure
from src.inference import run_inference_on_split
from src.grounding import DEFAULT_PROMPT

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# 1. Nạp model 4-bit
model, processor = load_model_and_processor(
    model_id=MODEL_ID,
    quantization="nf4-4bit",
    device_map={"": 0} if torch.cuda.is_available() else "auto",
    padding_side="left"
)

# 2. Kiểm tra cấu trúc và dtypes
struct = inspect_model_structure(model)
print(f"Tổng tham số: {struct['total_params']:,} | Dtypes: {struct['dtypes']}")

# 3. Chạy suy luận zero-shot trên 311 ảnh test
run_inference_on_split(
    model=model,
    processor=processor,
    split_jsonl="data/splits/test.jsonl",
    image_root="data",
    prompt=DEFAULT_PROMPT,
    output_file="work/e0.jsonl",
    batch_size=4,
    max_new_tokens=256
)


In [ ]:
# Đánh giá E0 bằng công cụ trọng tài chính thức scripts/score.py
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

Xây dựng hàm `encode_example()` sử dụng Chat Template của Qwen3-VL, mask `-100` cho toàn bộ prompt và visual tokens (chỉ tính loss trên token phản hồi JSON của assistant). Kiểm tra số lượng token vision (phải đúng 220 tokens cho ảnh 640x360).

In [ ]:
from src.dataset import RoadDamageDataset, encode_example, MultimodalDataCollator

train_dataset = RoadDamageDataset("data/splits/train.jsonl", image_root="data", prompt=DEFAULT_PROMPT)
val_dataset = RoadDamageDataset("data/splits/val.jsonl", image_root="data", prompt=DEFAULT_PROMPT)
print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Kiểm tra tokenization và loss masking trên 1 mẫu
sample = train_dataset[0]
enc = encode_example(sample["image"], sample["prompt"], sample["answer"], processor)
prefix_len = enc["prompt_len"].item()
print(f"Độ dài prompt prefix (bị mask -100): {prefix_len} tokens")

# Giải mã phần loss tokens (labels != -100)
loss_tokens = enc["labels"][enc["labels"] != -100]
decoded_loss = processor.tokenizer.decode(loss_tokens, skip_special_tokens=False)
print("Phần nhãn được tính loss:\n", decoded_loss)
print("✓ Kiểm tra Chat Template & Loss Masking: ĐẠT!")


## 5. Setup LoRA / đóng băng cho E1, E2, E3

Cấu hình LoRA adapter và unfreeze projector theo ma trận thí nghiệm:
- **E1**: LoRA LLM (252 modules) | Freeze ViT | Freeze Projector.
- **E2**: E1 + mở full Projector (`visual.merger`, `visual.deepstack_merger_list`) ở float32 (~109M tham số).
- **E3**: E2 + LoRA trên Attention và MLP của ViT (96 modules, tổng cộng 348 LoRA modules).

In [ ]:
from src.model_utils import apply_setup

print("Các hàm cấu hình E1, E2, E3 đã được triển khai đầy đủ trong src/model_utils.py.")
print("Mỗi setup sẽ được khởi tạo độc lập từ mô hình gốc để đảm bảo tính cô lập và khả năng tái lập thí nghiệm.")


## 6. Vòng huấn luyện

Chạy vòng lặp huấn luyện 300 optimizer steps với AdamW, cosine annealing scheduler, 5% warmup, mixed precision fp16 (GradScaler), gradient accumulation = 4, batch size = 2 (effective batch size 8), gradient checkpointing. Lưu checkpoint sau mỗi 100 bước sang Google Drive (cả adapter LoRA và `projector_weights.pt`).

In [ ]:
# Huấn luyện E1: LoRA LLM
import gc
from src.training import TrainConfig, train_experiment

gc.collect()
torch.cuda.empty_cache()

model_e1, _ = load_model_and_processor(MODEL_ID, quantization="nf4-4bit", device_map={"": 0}, padding_side="right")
model_e1 = apply_setup(model_e1, setup_name="E1", lora_r=16, lora_alpha=32, lora_dropout=0.05)

cfg_e1 = TrainConfig(
    name="E1",
    lr=2.0e-4,
    batch_size=2,
    grad_accum=4,
    max_steps=300,
    output_dir="work/checkpoints/e1",
    drive_dir=DRIVE_CHECKPOINT_DIR,
    train_projector=False
)
run_info_e1 = train_experiment(cfg_e1, model_e1, processor, train_dataset)
del model_e1
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Huấn luyện E2: LoRA LLM + Projector
gc.collect()
torch.cuda.empty_cache()

model_e2, _ = load_model_and_processor(MODEL_ID, quantization="nf4-4bit", device_map={"": 0}, padding_side="right")
model_e2 = apply_setup(model_e2, setup_name="E2", lora_r=16, lora_alpha=32, lora_dropout=0.05, train_projector=True)

cfg_e2 = TrainConfig(
    name="E2",
    lr=2.0e-4,
    batch_size=2,
    grad_accum=4,
    max_steps=300,
    output_dir="work/checkpoints/e2",
    drive_dir=DRIVE_CHECKPOINT_DIR,
    train_projector=True
)
run_info_e2 = train_experiment(cfg_e2, model_e2, processor, train_dataset)
del model_e2
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Huấn luyện E3: LoRA LLM + Projector + LoRA ViT
gc.collect()
torch.cuda.empty_cache()

model_e3, _ = load_model_and_processor(MODEL_ID, quantization="nf4-4bit", device_map={"": 0}, padding_side="right")
model_e3 = apply_setup(model_e3, setup_name="E3", lora_r=16, lora_alpha=32, lora_dropout=0.05, train_projector=True)

cfg_e3 = TrainConfig(
    name="E3",
    lr=2.0e-4,
    batch_size=2,
    grad_accum=4,
    max_steps=300,
    output_dir="work/checkpoints/e3",
    drive_dir=DRIVE_CHECKPOINT_DIR,
    train_projector=True
)
run_info_e3 = train_experiment(cfg_e3, model_e3, processor, train_dataset)
del model_e3
gc.collect()
torch.cuda.empty_cache()


## 7. Suy luận và chấm E1–E3

Nạp lại mô hình từ checkpoint (bao gồm LoRA adapter và file `projector_weights.pt` cho E2/E3), chạy suy luận trên 311 ảnh test và tính điểm bằng `scripts/score.py` với metadata bắt buộc (`trainable_params`, `peak_vram_gb`, `train_minutes`).

In [ ]:
from src.model_utils import load_experiment_checkpoint

experiments = [
    ("E1", "work/checkpoints/e1", False, "work/e1.jsonl", "report/evidence/e1.json"),
    ("E2", "work/checkpoints/e2", True, "work/e2.jsonl", "report/evidence/e2.json"),
    ("E3", "work/checkpoints/e3", True, "work/e3.jsonl", "report/evidence/e3.json"),
]

for exp_name, ckpt_dir, is_proj, pred_file, ev_file in experiments:
    print(f"\n{'='*60}\nĐánh giá {exp_name}...\n{'='*60}")
    meta_file = Path(ckpt_dir) / "run.json"
    run_meta = json.loads(meta_file.read_text(encoding="utf-8")) if meta_file.exists() else {}

    base_model, proc = load_model_and_processor(MODEL_ID, quantization="nf4-4bit", device_map={"": 0}, padding_side="left")
    eval_model = load_experiment_checkpoint(base_model, ckpt_dir, is_projector_trained=is_proj)

    run_inference_on_split(
        model=eval_model,
        processor=proc,
        split_jsonl="data/splits/test.jsonl",
        image_root="data",
        prompt=DEFAULT_PROMPT,
        output_file=pred_file,
        batch_size=4,
        max_new_tokens=256
    )

    del eval_model, base_model
    gc.collect()
    torch.cuda.empty_cache()

    params = run_meta.get("trainable_params", 0)
    vram = run_meta.get("peak_vram_gb", 0.0)
    time_min = run_meta.get("train_minutes", 0.0)

    !python scripts/score.py --pred {pred_file} --name {exp_name} --out {ev_file} \
        --meta trainable_params={params} --meta peak_vram_gb={vram} --meta train_minutes={time_min}


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

Thực hiện thí nghiệm loại trừ E4 (Ablation study trên biến data_size 10% vs 100%), kiểm chứng hàm reward IoU cho E5 GRPO, chạy nghiệm thu `check_submission.py` và sinh 6 ảnh minh hoạ kết quả.

In [ ]:
# E4: Ablation trên kích thước dữ liệu huấn luyện (10% vs 100% data)
gc.collect()
torch.cuda.empty_cache()

train_10pct = RoadDamageDataset("data/splits/train.jsonl", fraction=0.1, seed=42)
print(f"E4: Huấn luyện với 10% tập train ({len(train_10pct)} ảnh)...")

model_e4, _ = load_model_and_processor(MODEL_ID, quantization="nf4-4bit", device_map={"": 0}, padding_side="right")
model_e4 = apply_setup(model_e4, setup_name="E1", lora_r=16, lora_alpha=32, lora_dropout=0.05)

cfg_e4 = TrainConfig(
    name="E4_data_0.1",
    lr=2.0e-4,
    batch_size=2,
    grad_accum=4,
    max_steps=300,
    output_dir="work/checkpoints/e4_data_0.1",
    drive_dir=DRIVE_CHECKPOINT_DIR,
    train_projector=False
)
run_info_e4 = train_experiment(cfg_e4, model_e4, processor, train_10pct)
del model_e4
gc.collect()
torch.cuda.empty_cache()

# Suy luận E4 (10% data)
base_model, proc = load_model_and_processor(MODEL_ID, quantization="nf4-4bit", device_map={"": 0}, padding_side="left")
eval_model_e4 = load_experiment_checkpoint(base_model, "work/checkpoints/e4_data_0.1", is_projector_trained=False)
run_inference_on_split(eval_model_e4, proc, split_jsonl="data/splits/test.jsonl", output_file="work/e4_data_0.1.jsonl", batch_size=4)
del eval_model_e4, base_model
gc.collect()
torch.cuda.empty_cache()

params4 = run_info_e4["trainable_params"]
vram4 = run_info_e4["peak_vram_gb"]
time4 = run_info_e4["train_minutes"]
!python scripts/score.py --pred work/e4_data_0.1.jsonl --name E4_data_0.1 --out report/evidence/e4_data_0.1.json \
    --meta trainable_params={params4} --meta peak_vram_gb={vram4} --meta train_minutes={time4}

# E4 100% data: chính là E1 (cùng kiến trúc, seed, lr, steps)
import shutil
shutil.copy2("report/evidence/e1.json", "report/evidence/e4_data_1.0.json")
print("✓ Đã sinh bằng chứng cho cả 2 mức dữ liệu e4_data_0.1.json và e4_data_1.0.json!")


In [ ]:
# E5: Kiểm tra hàm Reward IoU cho GRPO và nghiệm thu check_submission.py
from src.reward import compute_grounding_iou_reward

gt_sample = [{"label": "pothole", "bbox": (100, 200, 300, 400)}]
pred_sample = json.dumps([{"label": "pothole", "bbox_2d": [100, 200, 300, 400]}])
rew = compute_grounding_iou_reward(pred_sample, gt_sample)
print(f"✓ Reward IoU kiểm chứng: {rew} (Kỳ vọng: 1.0)")

# Chạy nghiệm thu tự động
!python scripts/check_submission.py


In [ ]:
# Sinh 6 ảnh minh hoạ kết quả mô hình vào report/figures/
from src.visualize import draw_side_by_side_comparison
from scripts.score import parse_output

test_rows = read_jsonl("data/splits/test.jsonl")
e1_preds = {r["image_id"]: r["raw"] for r in read_jsonl("work/e1.jsonl")}
selected_ids = [test_rows[i]["image_id"] for i in [0, 5, 12, 20, 35, 50]]

for idx, img_id in enumerate(selected_ids, 1):
    entry = next(e for e in test_rows if e["image_id"] == img_id)
    raw_p = e1_preds.get(img_id, "[]")
    parsed_p = parse_output(raw_p) or []
    img_p = Path("data") / entry["image_path"]
    im = Image.open(img_p).convert("RGB")
    gt_f = [{
        "label": o["label"],
        "bbox": (
            (o["yolo"][0] - o["yolo"][2]/2)*1000,
            (o["yolo"][1] - o["yolo"][3]/2)*1000,
            (o["yolo"][0] + o["yolo"][2]/2)*1000,
            (o["yolo"][1] + o["yolo"][3]/2)*1000
        )
    } for o in entry["objects"]]
    fig_p = f"report/figures/prediction_{idx:02d}.png"
    draw_side_by_side_comparison(
        image=im,
        gt_boxes=gt_f,
        pred_boxes=parsed_p,
        title=f"Minh hoạ dự đoán #{idx:02d} - Image ID: {img_id}",
        subtitle=f"Ground Truth: {len(gt_f)} boxes | Dự đoán: {len(parsed_p)} boxes",
        save_path=fig_p
    )
print("✓ Đã lưu thành công 6 ảnh minh họa vào report/figures/")
